## Day 3 - Part 5: BERT, NLP의 판도를 바꾼 혁명가

### 개요

Day 3의 앞선 파트에서 우리는 순서가 있는 데이터를 처리하기 위해 RNN의 순환 구조를 버리고 '어텐션' 메커니즘을 도입한 트랜스포머(Transformer)의 구조를 탐험했습니다. 

트랜스포머는 병렬 처리와 장거리 의존성 문제 해결이라는 위대한 혁신을 이뤄냈습니다. 

하지만 이 강력한 모델을 처음부터 학습시키는 것은 엄청난 양의 데이터와 컴퓨팅 자원을 필요로 하는, 여전히 거대한 장벽이었습니다.

만약 누군가 이미 인터넷의 방대한 텍스트로 트랜스포머를 아주 똑똑하게 '미리 학습'시켜놓고, 우리는 그 지능을 가져와 우리의 작은 문제에 맞게 살짝만 '미세 조정'하여 사용할 수 있다면 어떨까요?

2018년, 구글 연구팀이 바로 이 아이디어를 현실로 만든 모델, `BERT(Bidirectional Encoder Representations from Transformers)`를 발표하며 자연어 처리(NLP) 세계에 지각 변동을 일으켰습니다.  

BERT는 특정 작업에 종속되지 않는, 범용 언어 이해 능력을 갖춘 최초의 고성능 사전 학습 모델이었습니다. 

마치 다양한 요리에 모두 쓸 수 있는 만능 육수처럼, BERT는 감성 분석, 질문 답변, 문장 관계 추론 등 11개 이상의 NLP 작업을 약간의 파인튜닝만으로 해결하며 기존의 모든 기록을 갈아치웠습니다. 

이번 파트에서는 현대 NLP의 표준이 된 '사전 학습 및 파인튜닝' 패러다임을 연 BERT의 심장부로 떠납니다. 

BERT가 어떻게 문장을 '양방향'으로 읽으며 진정한 문맥을 이해하는지, 어떤 '스스로 학습하는 과제'를 통해 언어의 구조를 터득하는지 그 비밀을 파헤쳐 봅니다. 

마지막으로, Hugging Face 라이브러리를 사용해 미리 학습된 BERT를 가져와, 네이버 영화 리뷰 데이터셋(NSMC)에 대한 감성 분석 모델을 직접 만들어보며 BERT의 막강한 힘을 직접 체험하게 될 것입니다.

`이번 파트의 학습 목표:`

  * NLP에서 `사전 학습(Pre-training)과 파인튜닝(Fine-tuning)` 패러다임의 중요성을 이해합니다.
  
  * BERT의 핵심 특징인 `양방향성(Bidirectionality)` 이 기존 모델과 어떻게 다른지, 왜 중요한지 설명할 수 있습니다.
  * BERT의 두 가지 사전 학습 과제인 `MLM(Masked Language Model)` 과 `NSP(Next Sentence Prediction)` 의 개념을 이해합니다.
  * 사전 학습된 BERT 모델에 새로운 층을 추가하여 특정 작업에 맞게 학습시키는 `파인튜닝 과정`을 설명할 수 있습니다.
  * `Hugging Face Transformers` 라이브러리를 사용하여 사전 학습된 BERT 모델과 토크나이저를 로드할 수 있습니다.
  * 네이버 영화 리뷰 데이터(NSMC)를 활용하여, BERT 모델을 파인튜닝하는 감성 분석 프로젝트 전 과정을 수행하고 그 결과를 분석할 수 있습니다.

### 1. 전이 학습의 시대: BERT의 등장

BERT 이전의 NLP 세계는 각 문제에 맞는 모델을 처음부터 설계하고 학습시키는 것이 일반적이었습니다. 

감성 분석 모델, 기계 번역 모델, 개체명 인식 모델이 모두 별개로 존재했죠.  

이는 마치 요리할 때마다 육수를 새로 내는 것과 같아 비효율적이었습니다.

컴퓨터 비전 분야에서는 이미 ImageNet 같은 대규모 데이터셋으로 미리 학습된 모델(예: ResNet, VGG)을 가져와 다른 이미지 관련 작업에 활용하는 `전이 학습(Transfer Learning)` 이 표준으로 자리 잡고 있었습니다.  

이 모델들은 이미지의 기본적인 특징(선, 질감, 모양 등)을 이미 알고 있으므로, 적은 데이터로도 특정 작업(예: 개/고양이 분류)에 빠르게 적응하여 높은 성능을 낼 수 있었습니다.

BERT는 바로 이 전이 학습의 성공을 NLP 세계로 가져왔습니다. 

  * `사전 학습 (Pre-training):` 위키피디아(25억 단어)와 BookCorpus(8억 단어) 같은 거대한 텍스트 데이터(총 33억 단어)를 사용하여 트랜스포머 인코더 기반의 모델을 미리 학습시킵니다.  이 과정에서 BERT는 특정 작업이 아닌, 언어 자체의 구조, 문법, 단어 간의 관계, 세상의 상식 등 범용적인 '언어 이해 능력'을 습득합니다. 
  
  * `파인튜닝 (Fine-tuning):` 이렇게 똑똑해진 BERT 위에, 우리가 풀고 싶은 특정 작업(예: 감성 분석)을 위한 작은 분류기(단순한 신경망 층)를 하나 올립니다. 그리고 우리가 가진 (상대적으로 훨씬 작은) 레이블링된 데이터로 모델 전체를 아주 조금만 더 학습시킵니다.  BERT는 이미 언어를 깊이 이해하고 있으므로, 몇 번의 학습(epoch)만으로도 새로운 작업에 빠르게 적응하여 놀라운 성능을 보여줍니다. 

이 '사전 학습 후 파인튜닝' 접근법 덕분에, 이제 우리는 거대한 데이터나 컴퓨팅 자원 없이도 최첨단 NLP 모델의 성능을 누릴 수 있게 되었습니다.

### 2. BERT는 세상을 어떻게 배우는가: 양방향성과 사전 학습

그렇다면 BERT는 어떻게 그토록 강력한 범용 언어 이해 능력을 갖추게 되었을까요? 그 비결은 '진정한 양방향성'과 '두 가지 독창적인 사전 학습 과제'에 있습니다.

#### 2.1. 진정한 문맥 이해: 양방향(Bidirectional) 모델

기존의 언어 모델(GPT나 RNN 등)은 문장을 왼쪽에서 오른쪽으로, 즉 단방향으로만 처리했습니다. "나는 오늘 은행에 가서..." 라는 문장에서 '은행'의 의미를 예측할 때, 오직 '나는 오늘'이라는 왼쪽의 정보만 활용할 수 있었죠.

하지만 단어의 의미는 양쪽 문맥에 의해 결정되는 경우가 많습니다.

1.  "나는 `은행`에 가서 대출을 받았다." (금융 기관)
2.  "강물이 넘쳐 `은행`이 모두 무너졌다." (강둑)

'은행'이라는 단어의 의미를 정확히 알려주는 단서는 '대출'이나 '강물'처럼 단어의 오른쪽에 나타납니다. 단방향 모델은 이 정보를 놓치기 쉽습니다.

BERT는 트랜스포머의 인코더 구조를 사용하여, 문장의 모든 단어를 `한 번에 동시에` 바라봅니다. 

이를 통해 각 단어의 의미를 파악할 때, 그 단어의 `왼쪽과 오른쪽 모든 문맥을 동시에 고려`합니다.  

이것이 바로 BERT가 '양방향(Bidirectional)'이라고 불리는 이유이며, 미묘한 언어의 뉘앙스까지 파악할 수 있는 핵심 비결입니다. 

#### 2.2. 스스로 학습하는 두 가지 과제: MLM과 NSP

BERT는 이 양방향 구조를 활용하여, 레이블이 없는 방대한 텍스트로부터 스스로 학습하는 두 가지 기발한 과제를 수행합니다. 이를 `자기 지도 학습(Self-supervised Learning)` 이라고 합니다. 

`1. MLM (Masked Language Model, 마스크 언어 모델)`

MLM은 문장에서 무작위로 단어를 가리고(마스킹하고), 그 가려진 단어가 무엇이었는지 맞추게 하는 과제입니다.  마치 영어 빈칸 채우기(Cloze) 문제와 같습니다.

> 원본 문장: "앨리스는 흰 토끼를 따라 이상한 나라로 갔다."
>
> MLM 입력: "앨리스는 흰 토끼를 따라 [MASK] 나라로 갔다."

BERT는 이 `[MASK]` 토큰에 들어갈 단어를 예측하기 위해, '앨리스', '흰 토끼', '나라' 등 문장의 양쪽 문맥을 모두 활용해야 합니다.  

이 과정을 수십억 번 반복하면서, BERT는 단어와 단어 사이의 관계, 문법 구조, 의미적 유사성 등을 자연스럽게 학습하게 됩니다. 

  * `학습 기법:` 훈련 데이터의 15% 단어를 무작위로 선택하여, 그 중 80%는 `[MASK]` 토큰으로 바꾸고, 10%는 다른 무작위 단어로 바꾸며, 10%는 원래 단어 그대로 둡니다.  이는 파인튜닝 단계에서는 `[MASK]` 토큰이 없기 때문에, 모델이 실제 단어에 대해서도 문맥적 표현을 잘 학습하도록 돕습니다.

`2. NSP (Next Sentence Prediction, 다음 문장 예측)`

NSP는 두 문장을 주고, 두 번째 문장이 첫 번째 문장 바로 다음에 이어지는 문장이 맞는지(IsNext) 아닌지(NotNext)를 맞추게 하는 과제입니다. 

> `[IsNext 예시]`
>
> 문장 A: "앨리스는 흰 토끼를 따라 굴 속으로 들어갔다." 
>
> 문장 B: "그녀는 이상한 방에 있는 자신을 발견했다." 
>
> `정답: IsNext` (두 문장은 자연스럽게 이어짐)
>
> -----
>
> `[NotNext 예시]`
>
> 문장 A: "앨리스는 흰 토끼를 따라 굴 속으로 들어갔다." 
>
> 문장 B: "네오는 빨간 약과 파란 약 중 하나를 선택하라는 요청을 받았다." 
>
> `정답: NotNext` (두 문장은 전혀 관련 없음)

이 NSP 과제를 통해, BERT는 문장과 문장 사이의 관계, 즉 담화의 일관성이나 주제의 연속성과 같은 더 큰 범위의 논리적 흐름을 학습하게 됩니다. 

이 두 가지(MLM, NSP) 사전 학습 과제를 통해 BERT는 단어 수준과 문장 수준의 언어 이해 능력을 동시에 기르게 되고, 이렇게 얻어진 깊은 언어 표현(representation)이 파인튜닝 시 강력한 힘을 발휘하는 것입니다. 

### 3. BERT 커스터마이징: 파인튜닝(Fine-Tuning)

사전 학습을 마친 똑똑한 BERT는 이제 우리의 특정 문제를 풀 준비가 되었습니다. 파인튜닝 과정은 놀랍도록 간단합니다. 

1.  `사전 학습된 BERT 모델 로드:` Hugging Face 같은 라이브러리를 통해 이미 학습이 완료된 BERT 모델을 불러옵니다.

2.  `작업별 레이어 추가:` 우리의 문제에 맞는 간단한 신경망 층을 BERT 모델의 맨 위에 추가합니다. 예를 들어, 긍정/부정 감성 분석(이진 분류)을 하고 싶다면, 2개의 출력 뉴런을 가진 완전연결층(Linear Layer) 하나만 추가하면 됩니다. 
3.  `작업별 데이터로 학습:` 우리가 가진 (레이블이 있는) 데이터로 모델 전체를 짧게(보통 2~4 에포크) 학습시킵니다.  이때 BERT의 기존 가중치들은 아주 조금만 변하고, 우리가 새로 추가한 층의 가중치가 집중적으로 학습됩니다.

분류 문제의 경우, BERT는 문장의 시작을 알리는 특별한 토큰인 `[CLS]`를 사용합니다. 

BERT는 이 `[CLS]` 토큰의 최종 출력 벡터에 문장 전체의 의미를 압축하여 담도록 학습됩니다.  

우리는 파인튜닝 시 이 `[CLS]` 토큰의 벡터를 가져와서, 그 위에 추가한 분류층에 입력으로 넣어 최종 예측(예: 긍정/부정 확률)을 얻습니다. 


### 4. BERT를 내 손으로: Hugging Face Transformers 라이브러리

과거에는 BERT와 같은 거대 모델을 사용하는 것이 매우 복잡했지만, `Hugging Face`라는 회사가 `transformers` 라이브러리를 오픈소스로 공개하면서 판도가 바뀌었습니다. 

이제 단 몇 줄의 코드만으로 수천 개의 사전 학습된 모델을 다운로드하고, 토크나이징하며, 파인튜닝할 수 있게 되었습니다.

이번 실습에서 우리는 Hugging Face의 다음 구성 요소들을 주로 사용하게 될 것입니다.

  * `datasets`: NSMC와 같은 공개 데이터셋을 쉽게 다운로드하고 처리할 수 있게 해줍니다.
  
  * `AutoTokenizer`: 텍스트를 BERT가 이해할 수 있는 숫자(토큰 ID) 시퀀스로 변환합니다. 모델에 맞는 토크나이저를 자동으로 찾아줍니다. 
  * `AutoModelForSequenceClassification`: 분류 작업용 헤드가 이미 부착된 사전 학습된 BERT 모델을 불러옵니다. 
  * `TrainingArguments` & `Trainer`: 복잡한 학습 루프, 배치 처리, 평가, 모델 저장 등을 대신 처리해주는 편리한 도구입니다. 

이제 이 강력한 도구들을 가지고, 직접 BERT 모델을 파인튜닝하는 실습을 진행해 보겠습니다.


### 5. 종합 실습: 네이버 영화 리뷰 감성 분석 (NSMC)

이번 종합 실습에서는 `네이버 영화 리뷰 데이터셋(NSMC)` 을 사용하여, 주어진 영화 리뷰가 `긍정(1)` 인지 `부정(0)` 인지를 분류하는 BERT 모델을 만들어 보겠습니다. 

NSMC는 15만 개의 훈련 데이터와 5만 개의 테스트 데이터로 구성된 한국어 데이터셋입니다. 

우리는 한국어를 포함한 104개 언어로 사전 학습된 `bert-base-multilingual-cased` 모델을 사용하여 이 문제를 해결할 것입니다. 

#### 5.1. 환경 설정 및 데이터 로드

먼저 실습에 필요한 라이브러리를 설치하고, Hugging Face Hub에서 NSMC 데이터셋을 직접 로드합니다.

In [1]:
# 필요한 라이브러리 설치 (코랩 또는 새 환경에서 실행)
!pip install transformers datasets evaluate accelerate

  Using cached transformers-4.52.4-py3-none-any.whl.metadata (38 kB)
  Using cached regex-2024.11.6-cp311-cp311-win_amd64.whl.metadata (41 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.3.2-py2.py3-none-any.whl.metadata (3.8 kB)
  Using cached frozenlist-1.7.0-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached propcache-0.3.2-cp311-cp311-win_amd64.whl.metadata (12 kB)
  Using cached yarl-1.20.1-cp311-cp311-win_amd64.whl.metadata (76 kB)
Using cached transformers-4.52.4-py3-none-any.whl (10.5 MB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 20.4 MB/s eta 0:00:00
Using cached yarl-1.20.1-cp311-cp311-win_amd64.whl (86 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached aiosignal-1.3.2-py2.py3-none-any.whl (7.6 kB)
Using cached frozenlist-1.7.0-cp311

In [2]:

import pandas as pd
import numpy as np
import plotly.express as px
from datasets import load_dataset


# 1. 훈련 데이터셋을 DataFrame으로 변환하여 미리보기
train_df = pd.read_csv("..//datasets/text/nsmc/ratings_train.txt", sep="\t")
test_df = pd.read_csv("..//datasets/text/nsmc/ratings_test.txt", sep="\t")

print("훈련 데이터 샘플 (처음 5개):")
train_df.head()


훈련 데이터 샘플 (처음 5개):


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
# 2. 데이터 라벨 분포 시각화
label_counts = train_df['label'].value_counts().reset_index()
label_counts.columns = ['label', 'count']
label_counts['label'] = label_counts['label'].map({0: '부정 (0)', 1: '긍정 (1)'})

# 레이블 데이터 분포
label_counts

,label,count
0,부정 (0),75173
1,긍정 (1),74827


In [4]:
print("데이터셋 예시:")
print(f"리뷰: {train_df.iloc[0]['document']}")
print(f"라벨: {train_df.iloc[0]['label']}")

데이터셋 예시:
리뷰: 아 더빙.. 진짜 짜증나네요 목소리
라벨: 0


#### 5.2. 텍스트 전처리: 토크나이저(Tokenizer) 적용

BERT는 원시 텍스트를 직접 처리할 수 없습니다. 텍스트를 모델이 이해할 수 있는 숫자 시퀀스로 변환하는 `토큰화(Tokenization)` 과정이 필요합니다.  우리는 `bert-base-multilingual-cased` 모델에 맞는 `AutoTokenizer`를 로드하여 이 과정을 수행합니다.

토크나이저는 다음과 같은 일을 합니다:

  * 문장을 더 작은 단위(서브워드, Subword)로 분할합니다. (예: "짜증나네요" -\> "짜증", "\#\#나", "\#\#네요")
  * 각 토큰을 고유한 정수 ID로 매핑합니다. 
  * 문장의 시작에 `[CLS]`, 끝에 `[SEP]` 같은 특수 토큰을 추가합니다.
  * 문장 길이를 통일하기 위해 짧은 문장 뒤에 `[PAD]` 토큰을 추가(패딩)하고, 긴 문장은 잘라냅니다(잘림, Truncation).

<!-- end list -->

In [5]:
from transformers import AutoTokenizer

# 1. 사전 학습된 모델의 토크나이저 로드
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 2. 토큰화 함수 정의
def tokenize_function(examples):
    # truncation=True: max_length를 초과하는 시퀀스를 잘라냅니다.
    # padding="max_length": 모든 시퀀스를 max_length에 맞춰 패딩합니다.
    # 해결책: None 값을 빈 문자열로 변환하고, 문자열이 아닌 경우 처리
    documents = examples["document"]
    
    # None 값이나 NaN 값을 빈 문자열로 변환
    if isinstance(documents, list):
        documents = [str(doc) if doc is not None else "" for doc in documents]
    else:
        documents = str(documents) if documents is not None else ""
    
    return tokenizer(documents, padding="max_length", truncation=True, max_length=128)

# 3. DataFrame을 Hugging Face Dataset으로 변환
from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# 4. 전체 데이터셋에 토큰화 함수 일괄 적용
# batched=True 옵션으로 더 빠르게 처리 가능
tokenized_train_datasets = train_dataset.map(tokenize_function, batched=True, batch_size=1000)
tokenized_test_datasets = test_dataset.map(tokenize_function, batched=True, batch_size=1000)

# 토큰화 결과 확인
print("토큰화 후 추가된 컬럼들:")
print(tokenized_train_datasets[0].keys())

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\Users\Admin\workspace\analysis_study\class\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/150000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

토큰화 후 추가된 컬럼들:
dict_keys(['id', 'document', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])


In [7]:
print("첫 번째 리뷰의 토큰화 결과 (input_ids 일부):")
# [CLS] 토큰 ID: 101, [SEP] 토큰 ID: 102
print(tokenized_train_datasets[0]['input_ids'][:20])

첫 번째 리뷰의 토큰화 결과 (input_ids 일부):
[101, 9519, 9074, 119005, 119, 119, 9708, 119235, 9715, 119230, 16439, 77884, 48549, 9284, 22333, 12692, 102, 0, 0, 0]


#### 5.3. 사전 학습된 BERT 모델 로드

이제 파인튜닝할 BERT 모델을 로드할 차례입니다. 우리는 감성 분석(분류) 문제를 풀 것이므로, 분류를 위한 완전연결층(classification head)이 미리 추가된 `AutoModelForSequenceClassification` 클래스를 사용합니다. 

`num_labels=2` 로 설정하여 긍정/부정 2개의 클래스를 예측하도록 지정합니다. 

In [8]:
from transformers import AutoModelForSequenceClassification

# 1. 시퀀스 분류를 위한 사전 학습 모델 로드
# num_labels=2: 출력 클래스의 수를 2개(긍정/부정)로 설정
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
print("로드된 BERT 모델 구조 (마지막 분류 레이어 확인):")
print(model.classifier)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


로드된 BERT 모델 구조 (마지막 분류 레이어 확인):
Linear(in_features=768, out_features=2, bias=True)


#### 5.4. 모델 훈련 (파인튜닝)

Hugging Face의 `Trainer` API를 사용하면 파인튜닝 과정을 매우 간단하게 구현할 수 있습니다.  먼저 `TrainingArguments`를 통해 학습률, 에포크, 배치 크기 등 훈련에 필요한 하이퍼파라미터를 정의합니다.

In [9]:
import numpy as np
import evaluate

from transformers import TrainingArguments, Trainer

# 1. 훈련 인자(Hyperparameters) 설정
training_args = TrainingArguments(
    output_dir="../models/bert/",              # 모델과 체크포인트가 저장될 디렉토리
    num_train_epochs=2,                  # 총 훈련 에포크 수
    per_device_train_batch_size=16,      # 훈련용 배치 크기
    per_device_eval_batch_size=64,       # 평가용 배치 크기
    learning_rate=2e-5,                  # 학습률 
    warmup_steps=500,                    # 학습률 워밍업 스텝 수
    weight_decay=0.01,                   # 가중치 감쇠
    logging_dir='../logs',                # 로그 저장 디렉토리
    logging_steps=1000,                  # 로그 출력 스텝 간격
    save_strategy="epoch",               # 매 에포크마다 모델 저장
    eval_strategy="epoch",               # 매 에포크마다 평가 수행 (save_strategy와 일치시켜야 함)
    load_best_model_at_end=True,         # 훈련 종료 후 최적 모델 로드 
)

In [ ]:
# 2. 평가 지표 설정 (정확도)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 3. Trainer 인스턴스 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_datasets,
    eval_dataset=tokenized_test_datasets,
    compute_metrics=compute_metrics,
)

# 4. 모델 파인튜닝 시작
print("모델 파인튜닝을 시작합니다...")
trainer.train()
print("모델 파인튜닝 완료!")

모델 파인튜닝을 시작합니다...


c:\Users\Admin\workspace\analysis_study\class\.venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


#### 5.5. 모델 평가 및 새로운 문장 예측

훈련이 완료된 후, `trainer.evaluate()`를 통해 테스트 데이터셋에 대한 최종 성능을 확인할 수 있습니다. NSMC에 대해 BERT를 파인튜닝하면 보통 90% 내외의 높은 정확도를 얻을 수 있습니다. 

마지막으로, 우리가 직접 작성한 문장을 모델이 어떻게 예측하는지 테스트하는 함수를 만들어 보겠습니다.

In [ ]:
import torch

# 1. 테스트셋에 대한 최종 평가
print("최종 모델 평가:")
eval_results = trainer.evaluate()
print(f"테스트셋 정확도: {eval_results['eval_accuracy']:.4f}")

# 2. 새로운 문장 예측을 위한 파이프라인 생성
# device=0: GPU 사용, device=-1: CPU 사용
# PyTorch 모델과 토크나이저를 직접 사용해도 동일한 결과를 얻을 수 있습니다.
from transformers import pipeline

sentiment_classifier = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# 3. 예측 테스트
my_reviews = [
    "배우들 연기력은 좋았지만 스토리가 너무 진부하고 지루했어요.",
    "내 인생 최고의 영화! 보는 내내 감동과 웃음이 끊이지 않았습니다.",
    "이걸 영화라고 만들었냐? 진짜 시간 아까움.",
    "영상미가 정말 아름답고, OST도 환상적이네요."
]

print("\n새로운 문장 예측 결과:")
for review in my_reviews:
    result = sentiment_classifier(review)
    # 결과가 {'label': 'LABEL_1', 'score': 0.99...} 형태로 나옴
    label = "긍정" if result[0]['label'] == 'LABEL_1' else "부정"
    print(f"리뷰: \"{review}\" -> 예측: {label} (신뢰도: {result[0]['score']:.2f})")

이것으로 우리는 방대한 텍스트로 미리 학습된 범용 언어 모델 BERT를 가져와, 한국어 영화 리뷰의 감성을 분석하는 전문 모델로 성공적으로 변신시켰습니다.